# 🎓 대입 정보 RAG 파이프라인

| 단계 | 내용 |
|------|------|
| **1. Ingest** | JSON 로드 → 임베딩 → FAISS 인덱스 저장 |
| **2. Search** | 쿼리 임베딩 → 필터 + 벡터 검색 |
| **3. QA** | 검색 결과 → Claude로 답변 생성 |
| **4. Evaluate** | QA 케이스셋 기반 MRR / Hits@K / AnswerAccuracy 측정 |

> **데이터 형식 (admission_data_v3.json)**  
> 각 row = `{doc_id, university, admission_type, track, major, info_type, content}`

---
## ⚙️ 환경 설정
처음 실행 시 한 번만 실행하세요.

In [1]:
# 패키지 설치
!pip install -q faiss-cpu sentence-transformers anthropic tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 56.5 MB/s eta 0:00:00


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전역 설정  ← 여기만 수정하세요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

# # Anthropic API 키 (QA / Evaluate 단계에서 사용)
# ANTHROPIC_API_KEY = "sk-ant-..."   # ← 본인 키 입력
# os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# 파일 경로
DATA_PATH   = "admission_data_v3.json"   # 업로드한 데이터
INDEX_DIR   = "rag_index"                # FAISS 인덱스 저장 폴더

# 임베딩 모델
EMBED_MODEL = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"   # 한국어 SBERT

# 검색 파라미터
TOP_K         = 10    # FAISS에서 가져올 후보 수
RETURN_K      = 5     # 최종 반환 수

# Claude 모델
CLAUDE_MODEL  = "claude-sonnet-4-20250514"
MAX_TOKENS    = 512

print("✅ 설정 완료")

✅ 설정 완료


---
## 1️⃣ Ingest — 임베딩 생성 & FAISS 인덱스 저장

In [3]:
# ── 1-1. 데이터 로드 ──────────────────────────────────────
import json

with open(DATA_PATH, encoding="utf-8") as f:
    docs = json.load(f)

print(f"📄 총 문서 수: {len(docs):,}개")

# info_type 분포 확인
from collections import Counter
dist = Counter(d["info_type"] for d in docs)
print("\n📊 info_type 분포:")
for k, v in dist.most_common():
    print(f"  {k:10s}: {v:4d}개")

📄 총 문서 수: 1,121개

📊 info_type 분포:
  모집인원      :  980개
  지원자격      :   41개
  제출서류      :   28개
  유의사항      :   24개
  수능최저      :   17개
  평가방법      :   16개
  기타        :   10개
  전형일정      :    5개


In [4]:
# ── 1-2. 임베딩 생성 ──────────────────────────────────────
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

print(f"⚡ 모델 로딩: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)

# 임베딩할 텍스트: content 그대로 사용
texts = [d["content"] for d in docs]

print(f"⚡ 임베딩 생성 중... ({len(texts):,}개)")
BATCH_SIZE = 128
all_embeddings = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding"):
    batch = texts[i : i + BATCH_SIZE]
    vecs  = embedder.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_embeddings.append(vecs)

embeddings = np.vstack(all_embeddings).astype("float32")
print(f"✅ 임베딩 완료: shape={embeddings.shape}")

⚡ 모델 로딩: snunlp/KR-SBERT-V40K-klueNLI-augSTS


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⚡ 임베딩 생성 중... (1,121개)


Embedding:   0%|          | 0/9 [00:00<?, ?it/s]

✅ 임베딩 완료: shape=(1121, 768)


In [5]:
# ── 1-3. FAISS 인덱스 빌드 & 저장 ────────────────────────
import faiss, os, pickle

os.makedirs(INDEX_DIR, exist_ok=True)

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # 내적 = 코사인 유사도 (정규화된 벡터)
index.add(embeddings)

# 인덱스 저장
faiss.write_index(index, f"{INDEX_DIR}/faiss.index")

# 문서 메타데이터 저장
with open(f"{INDEX_DIR}/docs.pkl", "wb") as f:
    pickle.dump(docs, f)

print(f"✅ FAISS 인덱스 저장 완료")
print(f"   - 벡터 수  : {index.ntotal:,}")
print(f"   - 차원     : {dim}")
print(f"   - 저장 위치: {INDEX_DIR}/")

✅ FAISS 인덱스 저장 완료
   - 벡터 수  : 1,121
   - 차원     : 768
   - 저장 위치: rag_index/


---
## 2️⃣ Search — 메타 필터 + 벡터 검색

In [6]:
# ── 2-1. 인덱스 & 모델 로드 (이미 메모리에 있으면 스킵) ──
import faiss, pickle
from sentence_transformers import SentenceTransformer

if "index" not in dir() or index is None:
    print("📂 인덱스 로딩 중...")
    index    = faiss.read_index(f"{INDEX_DIR}/faiss.index")
    with open(f"{INDEX_DIR}/docs.pkl", "rb") as f:
        docs = pickle.load(f)
    embedder = SentenceTransformer(EMBED_MODEL)
    print("✅ 로드 완료")
else:
    print("✅ 이미 메모리에 로드됨")

✅ 이미 메모리에 로드됨


In [7]:
# ── 2-2. 검색 함수 정의 ───────────────────────────────────
import numpy as np
from typing import Optional

def search(
    query: str,
    university:     Optional[str] = None,
    admission_type: Optional[str] = None,   # "수시" | "정시"
    track:          Optional[str] = None,
    info_type:      Optional[str] = None,   # "모집인원" | "지원자격" | ...
    top_k: int = TOP_K,
    return_k: int = RETURN_K,
    use_filter: bool = True,
) -> list[dict]:
    """
    메타 필터(pre-filter) → 벡터 유사도 검색 → 상위 return_k 반환
    """
    # 1) 필터 후보 인덱스 결정
    if use_filter and any([university, admission_type, track, info_type]):
        candidate_ids = [
            i for i, d in enumerate(docs)
            if (university     is None or d["university"]     == university)
            and (admission_type is None or d["admission_type"] == admission_type)
            and (track          is None or d.get("track")      == track)
            and (info_type      is None or d["info_type"]      == info_type)
        ]
    else:
        candidate_ids = list(range(len(docs)))

    if not candidate_ids:
        return []

    # 2) 쿼리 임베딩
    q_vec = embedder.encode([query], normalize_embeddings=True).astype("float32")

    # 3) 후보 벡터만 추출해서 검색
    cand_vecs = np.array(
        [index.reconstruct(i) for i in candidate_ids], dtype="float32"
    )
    scores = (cand_vecs @ q_vec.T).squeeze()          # (N,)

    if scores.ndim == 0:   # 후보가 1개일 경우
        scores = scores.reshape(1)

    n_ret  = min(return_k, len(candidate_ids))
    top_idx = np.argpartition(scores, -n_ret)[-n_ret:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    results = []
    for idx in top_idx:
        doc = docs[candidate_ids[idx]]
        results.append({**doc, "score": float(scores[idx])})

    return results


print("✅ search() 함수 정의 완료")

✅ search() 함수 정의 완료


In [8]:
# ── 2-3. 검색 데모 ────────────────────────────────────────
# 아래 파라미터를 바꿔가며 테스트하세요
DEMO_QUERY          = "연세대학교 수시 특기자전형 컴퓨터과학과 모집인원은?"
DEMO_UNIVERSITY     = "연세대학교"
DEMO_ADMISSION_TYPE = "수시"
DEMO_TRACK          = None          # ex) "학종" | None
DEMO_INFO_TYPE      = "모집인원"    # ex) "지원자격" | None

results = search(
    query          = DEMO_QUERY,
    university     = DEMO_UNIVERSITY,
    admission_type = DEMO_ADMISSION_TYPE,
    track          = DEMO_TRACK,
    info_type      = DEMO_INFO_TYPE,
    return_k       = 5,
)

print(f"🔍 쿼리: {DEMO_QUERY}")
print(f"📦 결과 {len(results)}개\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] {r['doc_id']}  (score={r['score']:.4f})")
    print(f"    {r['content'][:120]}")
    print()

🔍 쿼리: 연세대학교 수시 특기자전형 컴퓨터과학과 모집인원은?
📦 결과 5개

[1] 연세대학교_수시_0005  (score=0.7755)
    연세대학교 수시 특기자전형 융합공학과 모집인원은 20명이다.

[2] 연세대학교_수시_0004  (score=0.7685)
    연세대학교 수시 특기자전형 융합공학전공 모집인원은 20명이다.

[3] 연세대학교_수시_0001  (score=0.7595)
    연세대학교 수시 특기자전형 컴퓨터과학과 모집인원은 66명이다.

[4] 연세대학교_수시_0039  (score=0.7354)
    연세대학교 수시 융합공학전공 모집인원은 20명이다.

[5] 연세대학교_수시_0034  (score=0.7047)
    연세대학교 수시 시스템반도체공학과 모집인원은 100명이다.



---
## 3️⃣ QA — 검색 결과 기반 Claude 답변 생성

In [ ]:
# # ── 3-1. QA 함수 정의 ────────────────────────────────────
# import anthropic

# client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# QA_SYSTEM = """당신은 대학 입학처 전문 상담사입니다.
# 아래 검색된 입시 정보 문서를 바탕으로 질문에 정확하게 답하세요.

# 규칙:
# 1. 문서에 있는 정보만 사용하세요. 추측하지 마세요.
# 2. 문서에 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.
# 3. 숫자(모집인원, 등급 등)는 정확히 표기하세요.
# 4. 답변은 간결하게 핵심만 작성하세요."""

# def qa(
#     question: str,
#     university:     Optional[str] = None,
#     admission_type: Optional[str] = None,
#     track:          Optional[str] = None,
#     info_type:      Optional[str] = None,
#     return_k: int = RETURN_K,
#     verbose: bool = True,
# ) -> dict:
#     """
#     검색 → Claude 답변 생성
#     반환: {answer, sources, search_results}
#     """
#     # 1) 검색
#     results = search(
#         query=question,
#         university=university,
#         admission_type=admission_type,
#         track=track,
#         info_type=info_type,
#         return_k=return_k,
#     )

#     if not results:
#         return {"answer": "관련 문서를 찾지 못했습니다.", "sources": [], "search_results": []}

#     # 2) 컨텍스트 구성
#     context_parts = []
#     for i, r in enumerate(results, 1):
#         context_parts.append(f"[문서 {i}] (출처: {r['doc_id']})\n{r['content']}")
#     context = "\n\n".join(context_parts)

#     user_prompt = f"""## 참고 문서
# {context}

# ## 질문
# {question}"""

#     # 3) Claude 호출
#     response = client.messages.create(
#         model=CLAUDE_MODEL,
#         max_tokens=MAX_TOKENS,
#         system=QA_SYSTEM,
#         messages=[{"role": "user", "content": user_prompt}],
#     )
#     answer = response.content[0].text.strip()

#     if verbose:
#         print(f"🔍 질문: {question}")
#         print(f"\n📄 검색된 문서 ({len(results)}개):")
#         for i, r in enumerate(results, 1):
#             print(f"  [{i}] {r['doc_id']}  score={r['score']:.4f}")
#             print(f"       {r['content'][:80]}...")
#         print(f"\n💬 답변:\n{answer}")

#     return {
#         "answer":         answer,
#         "sources":        [r["doc_id"] for r in results],
#         "search_results": results,
#     }

# print("✅ qa() 함수 정의 완료")

✅ qa() 함수 정의 완료


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

MODEL_ID = "beomi/Llama-3-Open-Ko-8B-Instruct-Preview"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

qa_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("✅ AI 모델 로드 완료!")

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/143 [00:00<?, ?B/s]

✅ AI 모델 로드 완료!


In [10]:
from typing import Optional

QA_SYSTEM = """당신은 대학 입학처 전문 상담사입니다.
아래 검색된 입시 정보 문서를 바탕으로 질문에 정확하게 답하세요.

규칙:
1. 문서에 있는 정보만 사용하세요. 추측하지 마세요.
2. 문서에 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.
3. 숫자(모집인원, 등급 등)는 정확히 표기하세요.
4. 답변은 간결하게 핵심만 작성하세요."""

def qa(
    question: str,
    university:     Optional[str] = None,
    admission_type: Optional[str] = None,
    track:          Optional[str] = None,
    info_type:      Optional[str] = None,
    return_k: int = RETURN_K,
    verbose: bool = True,
) -> dict:
    # 1) 검색
    results = search(
        query=question,
        university=university,
        admission_type=admission_type,
        track=track,
        info_type=info_type,
        return_k=return_k,
    )

    if not results:
        return {"answer": "관련 문서를 찾지 못했습니다.", "sources": [], "search_results": []}

    # 2) 컨텍스트 구성
    context = "\n\n".join(
        f"[문서 {i}] (출처: {r['doc_id']})\n{r['content']}"
        for i, r in enumerate(results, 1)
    )

    # 3) 프롬프트 구성 → 추론
    messages = [
        {"role": "system", "content": QA_SYSTEM},
        {"role": "user",   "content": f"## 참고 문서\n{context}\n\n## 질문\n{question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    output = qa_pipe(
        prompt,
        max_new_tokens=512,
        do_sample=False,
        return_full_text=False,
    )
    answer = output[0]["generated_text"].strip()

    if verbose:
        print(f"🔍 질문: {question}")
        print(f"\n📄 검색된 문서 ({len(results)}개):")
        for i, r in enumerate(results, 1):
            print(f"  [{i}] {r['doc_id']}  score={r['score']:.4f}")
            print(f"       {r['content'][:80]}...")
        print(f"\n💬 답변:\n{answer}")

    return {
        "answer":         answer,
        "sources":        [r["doc_id"] for r in results],
        "search_results": results,
    }

print("✅ qa() 함수 정의 완료")

✅ qa() 함수 정의 완료


In [11]:
# ── 3-2. QA 데모 ──────────────────────────────────────────
result = qa(
    question       = "연세대학교 수시 특기자전형 컴퓨터과학과 모집인원은?",
    university     = "연세대학교",
    admission_type = "수시",
    info_type      = "모집인원",
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 질문: 연세대학교 수시 특기자전형 컴퓨터과학과 모집인원은?

📄 검색된 문서 (5개):
  [1] 연세대학교_수시_0005  score=0.7755
       연세대학교 수시 특기자전형 융합공학과 모집인원은 20명이다....
  [2] 연세대학교_수시_0004  score=0.7685
       연세대학교 수시 특기자전형 융합공학전공 모집인원은 20명이다....
  [3] 연세대학교_수시_0001  score=0.7595
       연세대학교 수시 특기자전형 컴퓨터과학과 모집인원은 66명이다....
  [4] 연세대학교_수시_0039  score=0.7354
       연세대학교 수시 융합공학전공 모집인원은 20명이다....
  [5] 연세대학교_수시_0034  score=0.7047
       연세대학교 수시 시스템반도체공학과 모집인원은 100명이다....

💬 답변:
해당 정보를 찾을 수 없습니다. (문서 3에 따르면 컴퓨터과학과 모집인원은 66명이지만, 문서 1, 2, 4에는 해당 정보가 없습니다. 따라서 정확한 답을 찾을 수 없습니다.


In [12]:
# 다른 질문 테스트
result2 = qa(
    question       = "한양대학교 수시 학종전형 지원자격 조건은?",
    university     = "한양대학교",
    admission_type = "수시",
    track          = "학종",
    info_type      = "지원자격",
)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 질문: 한양대학교 수시 학종전형 지원자격 조건은?

📄 검색된 문서 (1개):
  [1] 한양대학교_수시_0070  score=0.3969
       [한양대학교] [수시][학종전형] [지원자격]
다음의 어느 하나에 해당하는 사람으로 국내 특성화고교(전문계고교)･마이스터고를 졸업하고 산업체 근...

💬 답변:
한양대학교 수시 학종전형 지원자격은 국내 특성화고교(전문계고교)･마이스터고를 졸업하고 산업체 근무 경력이 3년 이상인 재직자로, ｢초·중등교육법시행령｣ 제29조제2항제14호에 따른 일반고등학교에 재학하는 동안 시·도 교육감이 ｢직업 교육훈련 촉진법｣에 따른 직업교육훈련기관 중 직업교육훈련위탁기관으로 선정한 기관에서 1년 이상의 직업교육훈련과정을 이수하고 해당 일반고등학교를 졸업한 사람, ｢초·중등교육법 시행령｣ 제90조제1항제10호에 따른 산업수요 맞춤형 고등학교를 졸업한 사람, ｢평생교육법｣ 제31조제2항에 따른 학력인정 평생교육시설 중 특성화고등학교 등에서 제공하는 것과 같은 교육과정을 이수한 사람, 구 분 세부사항 •국가·지방자치단체 및 공공단체(소속 직원의 경우) •4대보험 중 1개 이상 가입 사업체(창업·자영업자 포함) 산업체 •4대보험 가입대상 사업체가 아닌 1차 산업 종사자는 국가·지름자체가 발급하는 공적증명서(농지원부 등본)를 통해 확인되는 경우 인정함 •4대보험 미가입 영세창업/자영업자는 사업자등록증을 소지하고, 세금 체납 사실이 없는 경우에 한해 인정함 •2025년 3월 1일 기준으로 총 재직기간이 3년 이상(예정)이어야 함 등으로 정의됩니다. 이 조건에 해당하는 지원자는 누구입니까?


---
## 4️⃣ Evaluate — MRR / Hits@K / AnswerAccuracy 평가

### QA 케이스셋 형식
```json
[
  {
    "question": "연세대학교 수시 특기자 컴퓨터과학과 모집인원은?",
    "answer": "66명",
    "relevant_doc_ids": ["연세대학교_수시_0001"],
    "university": "연세대학교",
    "admission_type": "수시",
    "track": "특기자",
    "info_type": "모집인원"
  },
  ...
]
```

In [13]:
# ── 4-1. 평가 케이스셋 준비 ──────────────────────────────
import json, os

QA_CASE_PATH = "qa_cases.json"   # 케이스셋 파일 경로

# ── 케이스셋 파일이 없으면 샘플로 자동 생성 ──
if not os.path.exists(QA_CASE_PATH):
    print("⚠️  qa_cases.json 없음 → 샘플 케이스셋 자동 생성")

    # 각 info_type에서 최대 5개씩 샘플링해서 케이스 생성
    from collections import defaultdict
    import random

    random.seed(42)
    by_type = defaultdict(list)
    for d in docs:
        by_type[d["info_type"]].append(d)

    # 케이스 템플릿
    INFO_QUESTION = {
        "모집인원": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"{d.get('major','')} 모집인원 수는?"
        ),
        "지원자격": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"지원자격 조건은?"
        ),
        "수능최저": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"수능 최저학력기준은?"
        ),
        "제출서류": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"제출서류 목록은?"
        ),
        "평가방법": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"평가 방법은?"
        ),
    }

    sample_cases = []
    for info_type, fn in INFO_QUESTION.items():
        pool = by_type[info_type]
        for d in random.sample(pool, min(5, len(pool))):
            sample_cases.append({
                "question":         fn(d),
                "answer":           "",          # 정답 레이블 (비워두면 AnswerAccuracy 스킵)
                "relevant_doc_ids": [d["doc_id"]],
                "university":       d["university"],
                "admission_type":   d["admission_type"],
                "track":            d.get("track"),
                "info_type":        info_type,
            })

    with open(QA_CASE_PATH, "w", encoding="utf-8") as f:
        json.dump(sample_cases, f, ensure_ascii=False, indent=2)
    print(f"✅ {len(sample_cases)}개 샘플 케이스 저장 → {QA_CASE_PATH}")
    print("   answer 필드를 채워두면 AnswerAccuracy도 평가됩니다.")

with open(QA_CASE_PATH, encoding="utf-8") as f:
    qa_cases = json.load(f)

print(f"\n📋 케이스 수: {len(qa_cases)}개")
from collections import Counter
print("유형별:", dict(Counter(c["info_type"] for c in qa_cases)))

⚠️  qa_cases.json 없음 → 샘플 케이스셋 자동 생성
✅ 25개 샘플 케이스 저장 → qa_cases.json
   answer 필드를 채워두면 AnswerAccuracy도 평가됩니다.

📋 케이스 수: 25개
유형별: {'모집인원': 5, '지원자격': 5, '수능최저': 5, '제출서류': 5, '평가방법': 5}


In [14]:
# ── 4-2. 검색 품질 평가 (MRR / Hits@K / nDCG) ────────────
import numpy as np
from tqdm.auto import tqdm

# 평가 파라미터
EVAL_TOP_K    = 10
EVAL_RETURN_K = 10
USE_FILTER    = True

def ndcg_at_k(relevant_set, ranked_ids, k):
    dcg  = sum(
        1 / np.log2(i + 2)
        for i, doc_id in enumerate(ranked_ids[:k])
        if doc_id in relevant_set
    )
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

# ── 배치 임베딩으로 속도 최적화 ──
queries = [c["question"] for c in qa_cases]
print(f"⚡ 배치 임베딩 생성 중... ({len(queries)}개 쿼리)")
q_vecs = embedder.encode(queries, normalize_embeddings=True,
                          batch_size=64, show_progress_bar=True).astype("float32")

# ── 케이스별 평가 ──
mrr_scores, hits1, hits3, hits5, hits10 = [], [], [], [], []
ndcg5_scores, ndcg10_scores = [], []
results_per_case = []

by_type = {}

for i, case in enumerate(tqdm(qa_cases, desc="Retrieval Eval")):
    relevant = set(case["relevant_doc_ids"])

    # 필터 후보
    if USE_FILTER:
        candidate_ids = [
            j for j, d in enumerate(docs)
            if (not case.get("university")     or d["university"]     == case["university"])
            and (not case.get("admission_type") or d["admission_type"] == case["admission_type"])
            and (not case.get("track")          or d.get("track")      == case["track"])
            and (not case.get("info_type")      or d["info_type"]      == case["info_type"])
        ]
    else:
        candidate_ids = list(range(len(docs)))

    if not candidate_ids:
        results_per_case.append([])
        for lst in [mrr_scores, hits1, hits3, hits5, hits10, ndcg5_scores, ndcg10_scores]:
            lst.append(0.0)
        continue

    cand_vecs = np.array([index.reconstruct(j) for j in candidate_ids], dtype="float32")
    scores    = (cand_vecs @ q_vecs[i].reshape(-1, 1)).squeeze()
    if scores.ndim == 0: scores = scores.reshape(1)

    n_ret    = min(EVAL_RETURN_K, len(candidate_ids))
    top_idx  = np.argpartition(scores, -n_ret)[-n_ret:]
    top_idx  = top_idx[np.argsort(scores[top_idx])[::-1]]
    ranked   = [docs[candidate_ids[j]]["doc_id"] for j in top_idx]

    results_per_case.append(ranked)

    # MRR
    rr = 0.0
    for rank, doc_id in enumerate(ranked, 1):
        if doc_id in relevant:
            rr = 1.0 / rank
            break
    mrr_scores.append(rr)

    # Hits@K
    hits1.append(float(any(d in relevant for d in ranked[:1])))
    hits3.append(float(any(d in relevant for d in ranked[:3])))
    hits5.append(float(any(d in relevant for d in ranked[:5])))
    hits10.append(float(any(d in relevant for d in ranked[:10])))

    # nDCG
    ndcg5_scores.append(ndcg_at_k(relevant, ranked, 5))
    ndcg10_scores.append(ndcg_at_k(relevant, ranked, 10))

    # 유형별 집계
    t = case.get("info_type", "기타")
    if t not in by_type:
        by_type[t] = {"hits5": [], "mrr": []}
    by_type[t]["hits5"].append(hits5[-1])
    by_type[t]["mrr"].append(rr)

# ── 결과 출력 ──
print()
print("📊 RAG 평가 리포트")
print("=" * 62)
print(f"  평가 케이스 수 : {len(qa_cases)}개")
print(f"  필터 사용      : {'ON' if USE_FILTER else 'OFF'}")
print()
print("┌─ 검색 품질 " + "─" * 38 + "┐")
print(f"│  MRR       : {np.mean(mrr_scores):.4f}   (평균 역순위, 높을수록 상위 노출)")
print(f"│  Hits@1    : {np.mean(hits1):.4f}  ({np.mean(hits1)*100:.1f}%)  1위가 정답인 비율")
print(f"│  Hits@3    : {np.mean(hits3):.4f}  ({np.mean(hits3)*100:.1f}%)  3위 안에 정답")
print(f"│  Hits@5    : {np.mean(hits5):.4f}  ({np.mean(hits5)*100:.1f}%)  5위 안에 정답")
print(f"│  Hits@10   : {np.mean(hits10):.4f}  ({np.mean(hits10)*100:.1f}%)  10위 안에 정답")
print(f"│  nDCG@5    : {np.mean(ndcg5_scores):.4f}   (순위 감쇠 반영 품질)")
print(f"│  nDCG@10   : {np.mean(ndcg10_scores):.4f}   (순위 감쇠 반영 품질)")
print("└" + "─" * 50 + "┘")

overall_mrr = np.mean(mrr_scores)
overall_h5  = np.mean(hits5)
if overall_mrr >= 0.6 and overall_h5 >= 0.75:
    judgment = "🟢 양호"
elif overall_mrr >= 0.45 and overall_h5 >= 0.60:
    judgment = "🟡 보통 (개선 권장)"
else:
    judgment = "🔴 미흡 (개선 필요)"
print(f"\n  → 종합 판정: {judgment}")

print()
print("┌─ 유형별 Hits@5 / MRR " + "─" * 28 + "┐")
for t, v in sorted(by_type.items(), key=lambda x: np.mean(x[1]["hits5"]), reverse=True):
    h5  = np.mean(v["hits5"])
    mrr = np.mean(v["mrr"])
    n   = len(v["hits5"])
    flag = "⚠️ " if h5 < 0.4 else "   "
    print(f"│  {flag}[{t:8s}]  Hits@5={h5:.3f}  MRR={mrr:.3f}  (n={n})")
print("└" + "─" * 50 + "┘")

# Hits@5=0 케이스 샘플
fail_cases = [
    (qa_cases[i], results_per_case[i])
    for i in range(len(qa_cases))
    if hits5[i] == 0
]
if fail_cases:
    print(f"\n⚠️  Hits@5=0 케이스: {len(fail_cases)}개 / {len(qa_cases)}개 (샘플 5개)")
    for case, ranked in fail_cases[:5]:
        print(f"   [{case['info_type']}] {case['question']}")
        print(f"        정답={case['relevant_doc_ids']}  |  검색상위={ranked[:3]}")

⚡ 배치 임베딩 생성 중... (25개 쿼리)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieval Eval:   0%|          | 0/25 [00:00<?, ?it/s]


📊 RAG 평가 리포트
  평가 케이스 수 : 25개
  필터 사용      : ON

┌─ 검색 품질 ──────────────────────────────────────┐
│  MRR       : 0.8433   (평균 역순위, 높을수록 상위 노출)
│  Hits@1    : 0.7200  (72.0%)  1위가 정답인 비율
│  Hits@3    : 0.9600  (96.0%)  3위 안에 정답
│  Hits@5    : 1.0000  (100.0%)  5위 안에 정답
│  Hits@10   : 1.0000  (100.0%)  10위 안에 정답
│  nDCG@5    : 0.8834   (순위 감쇠 반영 품질)
│  nDCG@10   : 0.8834   (순위 감쇠 반영 품질)
└──────────────────────────────────────────────────┘

  → 종합 판정: 🟢 양호

┌─ 유형별 Hits@5 / MRR ────────────────────────────┐
│     [모집인원    ]  Hits@5=1.000  MRR=0.750  (n=5)
│     [지원자격    ]  Hits@5=1.000  MRR=1.000  (n=5)
│     [수능최저    ]  Hits@5=1.000  MRR=0.800  (n=5)
│     [제출서류    ]  Hits@5=1.000  MRR=0.867  (n=5)
│     [평가방법    ]  Hits@5=1.000  MRR=0.800  (n=5)
└──────────────────────────────────────────────────┘


In [15]:
# ── 4-3. 답변 품질 평가 (AnswerAccuracy) ─────────────────
# answer 필드가 채워진 케이스만 평가합니다

labeled_cases = [c for c in qa_cases if c.get("answer", "").strip()]

if not labeled_cases:
    print("⚠️  answer 레이블이 없습니다. qa_cases.json의 answer 필드를 채워주세요.")
else:
    print(f"📋 답변 평가 케이스: {len(labeled_cases)}개")

    JUDGE_SYSTEM = """당신은 입시 정보 RAG 시스템의 답변 품질을 평가하는 심사관입니다.
주어진 질문, 정답(ground truth), AI 답변을 보고 아래 기준으로 평가하세요.

평가 기준:
- correct  : 정답의 핵심 정보가 AI 답변에 정확히 포함됨
- partial  : 부분적으로 맞지만 중요한 정보가 빠지거나 부정확함
- incorrect: 정답과 다르거나 핵심 정보가 틀림
- no_answer: AI가 답변을 거부하거나 "찾을 수 없다"고만 함

반드시 correct / partial / incorrect / no_answer 중 하나만 한 단어로 출력하세요."""

    correct_count = 0
    no_ans_count  = 0
    judge_results = []

    for case in tqdm(labeled_cases, desc="Answer Accuracy"):
        # 답변 생성
        out = qa(
            question       = case["question"],
            university     = case.get("university"),
            admission_type = case.get("admission_type"),
            track          = case.get("track"),
            info_type      = case.get("info_type"),
            verbose        = False,
        )

        # LLM-as-Judge
        judge_prompt = f"""질문: {case['question']}
정답(ground truth): {case['answer']}
AI 답변: {out['answer']}"""

        judge_resp = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=10,
            system=JUDGE_SYSTEM,
            messages=[{"role": "user", "content": judge_prompt}],
        )
        verdict = judge_resp.content[0].text.strip().lower()

        if verdict == "correct":
            correct_count += 1
        elif verdict == "no_answer":
            no_ans_count += 1

        judge_results.append({
            "question": case["question"],
            "gt":       case["answer"],
            "answer":   out["answer"],
            "verdict":  verdict,
        })

    acc     = correct_count / len(labeled_cases)
    no_rate = no_ans_count  / len(labeled_cases)
    acc_flag = "🟢" if acc >= 0.7 else ("🟡" if acc >= 0.5 else "🔴")

    print()
    print("┌─ QA 답변 품질 " + "─" * 36 + "┐")
    print(f"│  AnswerAccuracy : {acc:.4f}  ({acc*100:.1f}%)  → {acc_flag} {'양호' if acc >= 0.7 else '보통' if acc >= 0.5 else '미흡'}")
    print(f"│  NoAnswerRate   : {no_rate:.4f}  ({no_rate*100:.1f}%)  (낮을수록 좋음)")
    print(f"│  평가 케이스    : {len(labeled_cases)}개")
    print("└" + "─" * 50 + "┘")

    # 오답 샘플
    wrong = [r for r in judge_results if r["verdict"] not in ("correct",)]
    if wrong:
        print(f"\n❌ 오답/무응답 샘플 (최대 3개):")
        for r in wrong[:3]:
            print(f"  Q: {r['question']}")
            print(f"  GT: {r['gt']}")
            print(f"  AI: {r['answer'][:120]}")
            print(f"  판정: {r['verdict']}\n")

⚠️  answer 레이블이 없습니다. qa_cases.json의 answer 필드를 채워주세요.


In [16]:
# ── 4-4. 평가 결과 저장 ───────────────────────────────────
import json
from datetime import datetime

eval_output = {
    "timestamp": datetime.now().isoformat(),
    "n_cases":   len(qa_cases),
    "filter":    USE_FILTER,
    "retrieval": {
        "MRR":    round(float(np.mean(mrr_scores)), 4),
        "Hits@1": round(float(np.mean(hits1)),  4),
        "Hits@3": round(float(np.mean(hits3)),  4),
        "Hits@5": round(float(np.mean(hits5)),  4),
        "Hits@10":round(float(np.mean(hits10)), 4),
        "nDCG@5": round(float(np.mean(ndcg5_scores)), 4),
        "nDCG@10":round(float(np.mean(ndcg10_scores)), 4),
    },
    "by_type": {
        t: {
            "hits5": round(float(np.mean(v["hits5"])), 4),
            "mrr":   round(float(np.mean(v["mrr"])),   4),
            "n":     len(v["hits5"]),
        }
        for t, v in by_type.items()
    },
    "case_results": [
        {
            "question":     qa_cases[i]["question"],
            "info_type":    qa_cases[i].get("info_type"),
            "relevant":     qa_cases[i]["relevant_doc_ids"],
            "top5_ranked":  results_per_case[i][:5],
            "mrr":          round(mrr_scores[i], 4),
            "hit5":         int(hits5[i]),
        }
        for i in range(len(qa_cases))
    ]
}

SAVE_PATH = "eval_result.json"
with open(SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(eval_output, f, ensure_ascii=False, indent=2)

print(f"✅ 평가 결과 저장 완료: {SAVE_PATH}")

✅ 평가 결과 저장 완료: eval_result.json
